In [1]:
from losses import completion_network_loss, noise_loss
from utils import *
from classify import *
from generator import *
from discri import *
from torch.utils.data import DataLoader
from torch.optim import Adadelta, Adam
from torch.nn import BCELoss, DataParallel
from torchvision.utils import save_image
from torch.autograd import grad
import torchvision.transforms as transforms
import torch
import time
import random
import os, logging
import numpy as np
from attack import inversion, dist_inversion
from generator import Generator
from argparse import ArgumentDefaultsHelpFormatter, ArgumentParser
import torch, os, time, random, generator, discri, classify, utils
import numpy as np 
import torch.nn as nn
import torchvision.utils as tvls
import torch.nn.functional as F
from utils import log_sum_exp, save_tensor_images
from torch.autograd import Variable
import torch.optim as optim
import torch.autograd as autograd
import statistics 
import torch.distributions as tdist

# Load models
z_dim = 100
G = Generator(z_dim)
G = torch.nn.DataParallel(G).cuda()
path_G = './improvedGAN/improved_celeba_G.tar'
ckp_G = torch.load(path_G)
G.load_state_dict(ckp_G['state_dict'], strict=False)

T = VGG16(1000)
path_T = './target_model/target_ckp/VGG16_88.26.tar'
T = torch.nn.DataParallel(T).cuda()
ckp_T = torch.load(path_T)
T.load_state_dict(ckp_T['state_dict'], strict=False)


# initalize a sample of recovery script:
iden = torch.from_numpy(np.arange(60))
iden = iden.view(-1).long().cuda()

# get evaluation metric for backprop:
criterion = nn.CrossEntropyLoss().cuda()

bs = iden.shape[0]
no = torch.zeros(bs) # index for saving all success attack images
mu = Variable(torch.zeros(bs, 100), requires_grad=True)
log_var = Variable(torch.ones(bs, 100), requires_grad=True)

G.eval()
T.eval()

def reparameterize(mu, logvar):
    """
    Reparameterization trick to sample from N(mu, var) from
    N(0,1).
    :param mu: (Tensor) Mean of the latent Gaussian [B x D]
    :param logvar: (Tensor) Standard deviation of the latent Gaussian [B x D]
    :return: (Tensor) [B x D]
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)

    return eps * std + mu



/tmp/code/data_protection_tech_assg_1/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/tmp/code/data_protection_tech_assg_1/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
z = reparameterize(mu, log_var)
fake = G(z)
out = T(fake)[-1]

In [42]:
iden

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59], device='cuda:0')

In [10]:
z.shape

torch.Size([60, 100])

In [44]:
fake.shape

torch.Size([60, 3, 64, 64])

In [45]:
out_pred = T(fake)
print(len(out_pred[0]), len(out_pred[1]))

60 60


In [46]:
out.shape # 60x1000.    identities, containing 1000 guesses?

torch.Size([60, 1000])

In [7]:
# one identity sample
out[0][:10]

tensor([ 3.4580, -2.4421,  1.4437, -1.7843, -5.1658,  5.3803, -2.3547, -2.4328,
         0.8118,  0.5611], device='cuda:0', grad_fn=<SliceBackward0>)

In [27]:
loss_base = criterion(out, iden)
loss_base

tensor(12.8592, device='cuda:0', grad_fn=<NllLossBackward0>)

## Use SoftMax at first to mimic a black box model

In [25]:
sf = nn.Softmax(dim=1)

# input = torch.randn(2, 3)
# input
out_soft = sf(out)
out_soft[0][:10]

tensor([3.1365e-04, 8.5908e-07, 4.1844e-05, 1.6585e-06, 5.6385e-08, 2.1443e-03,
        9.3753e-07, 8.6716e-07, 2.2244e-05, 1.7311e-05], device='cuda:0',
       grad_fn=<SliceBackward0>)

In [28]:
loss_bb = criterion(out_soft, iden)
loss_bb


tensor(6.9072, device='cuda:0', grad_fn=<NllLossBackward0>)

In [29]:
out_soft_1d = sf(out)
max(out_soft[0])

tensor(0.3682, device='cuda:0', grad_fn=<UnbindBackward0>)

### Black box single label asnwer

In [31]:
out_soft_max = out_soft
max_idx = out_soft_max.argmax(dim=1)

out_soft_max.zero_()
out_soft_max.scatter_(1, max_idx.unsqueeze(1), 1)
out_soft_max[0][:10]
print(min(out_soft_max[0]), max(out_soft_max[0]))

tensor(0., device='cuda:0', grad_fn=<UnbindBackward0>) tensor(1., device='cuda:0', grad_fn=<UnbindBackward0>)
